In [42]:
import pandas as pd
import numpy as np
import yfinance as yf
import plotly.graph_objects as go

In [43]:
df = yf.download("QQQ", period="2y", interval="1d")
df.columns = df.columns.get_level_values(0)
df = df.reset_index()
df['Date'] = pd.to_datetime(df['Date'])
del df['Volume']
df


[*********************100%***********************]  1 of 1 completed


Price,Date,Close,High,Low,Open
0,2024-11-05,487.278351,487.941651,482.635339,482.724434
1,2024-11-06,500.514374,501.336075,494.594309,495.544682
2,2024-11-07,508.394623,509.176746,503.246742,503.306138
3,2024-11-08,508.988617,509.760770,507.275909,507.899602
4,2024-11-11,508.691711,510.414268,505.800954,510.206350
...,...,...,...,...,...
496,2026-10-29,715.340027,718.070007,711.619995,716.539978
497,2026-10-30,690.489990,698.630005,683.450012,697.210022
498,2026-11-02,681.090027,699.429993,680.330017,695.320007
499,2026-11-03,685.280029,688.690002,683.179993,684.390015


In [44]:
df1 = df.copy()
x = np.arange(len(df1))

mid_price = ((df1["High"] + df1["Low"]) / 2).to_numpy()
m, b = np.polyfit(x, mid_price, 1)
df1['midline'] = m * x + b

n = 2
residual = mid_price - df1['midline']
deviation = np.nanstd(residual)
df1['upper'] = df1['midline'] + n * deviation
df1['lower'] = df1['midline']- n * deviation

std = df1['midline'].std()
df1['upper1'] = df1['midline'] + n * std
df1['lower1'] = df1['midline'] - n * std

In [45]:
df1.set_index('Date', inplace=True)

In [46]:

fig = go.Figure(data=[go.Candlestick(x=df1.index,
                open=df1['Open'],
                high=df1['High'],
                low=df1['Low'],
                close=df1['Close'],
                increasing_line_color = 'rgba(19,156,19,0.8)',
                decreasing_line_color = 'rgba(175,07,49,0.8)',
                name = 'QQQ')])

fig.update_layout(
    annotations=[dict(text="Dynamic Trend Channels (2y)",
                    font=dict(color="white", size=16),
                    xref="paper",
                    yref="paper",
                    x=0.10,
                    y=1.05,
                    showarrow=False)])

fig.add_trace(go.Scatter(x=df1.index, 
                         y=df1.midline, 
                         opacity=0.8, 
                         line=dict(color='white', width=1),
                        name='MIDLINE'))

fig.add_trace(go.Scatter(x=df1.index, 
                         y=df1.upper, 
                         opacity=0.8, 
                         line=dict(color='white', width=1),
                        name='RES_UPPER'))

fig.add_trace(go.Scatter(x=df1.index, 
                         y=df1.lower, 
                         opacity=0.8, 
                         line=dict(color='white', width=1),
                        name='RES_LOWER'))

# fig.add_trace(go.Scatter(x=df1.index, 
#                          y=df1.upper1, 
#                          opacity=0.8, 
#                          line=dict(color='red', width=1),
#                         name='STD_UPPER'))

# fig.add_trace(go.Scatter(x=df1.index, 
#                          y=df1.lower1, 
#                          opacity=0.8, 
#                          line=dict(color='red', width=1),
#                         name='STD_LOWER'))

fig.update_layout(autosize=False, width=1000, height=600, 
                  xaxis_rangeslider_visible=False, 
                  template="plotly_dark")

fig.update_yaxes(gridcolor="#171717") 
fig.update_xaxes(gridcolor="#171717")

fig.update_xaxes(
    rangebreaks=[
        dict(bounds=["sat", "mon"]),
        #dict(bounds=[21, 13.5], pattern="hour")
    ])

#fig.write_image('2ychart.png', scale=2)

fig.show()

In [47]:

np.random.seed(0)
x = np.arange(20)
true_line = 2 * x + 10
noise = np.random.normal(0, 4, len(x))
y = true_line + noise  

m, b = np.polyfit(x, y, 1)
midline = m * x + b

resid = y - midline

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=x, y=y,
    mode="markers",
    marker=dict(color="red", size=8),
    name="Mid Price Values"
))

fig.add_trace(go.Scatter(
    x=x, y=midline,
    mode="lines",
    line=dict(color="white", width=2),
    name="Line of Best Fit"
))

for xi, yi, fi in zip(x, y, midline):
    fig.add_trace(go.Scatter(
        x=[xi, xi],
        y=[fi, yi],
        mode="lines",
        line=dict(color="gray", width=1, dash="dot"),
        showlegend=False
    ))

fig.update_layout(
    annotations=[dict(text="Residuals - Difference between Price and Best Fit Line ",
                    font=dict(color="white", size=14),
                    xref="paper",
                    yref="paper",
                    x=0.10,
                    y=1.09,
                    showarrow=False)])

fig.update_layout(autosize=False, width=1000, height=600, 
                  xaxis_rangeslider_visible=False, 
                  template="plotly_dark")

fig.update_yaxes(gridcolor="#171717") 
fig.update_xaxes(gridcolor="#171717")

#fig.write_image('residuals.png', scale=2)

fig.show()

In [48]:

window = 86           
frame_rate = 80       
n = 2              

frames = []
          
for i in range(window-1, len(df1)):
    w = df1.iloc[i - window + 1 : i + 1]

    x = np.arange(window)
    mid_price = ((w['High'] + w['Low'])/2).to_numpy()
    m, b = np.polyfit(x, mid_price, 1)
    midline = m*x + b

    residuals = mid_price - midline
    deviation = np.nanstd(residuals)                         
    std = np.nanstd(midline)                     

    up_res = midline + n * deviation
    dn_res = midline - n * deviation
    up_line = midline + std
    dn_line = midline - std

    y_min = float(min(w['Low'].min(),  dn_res.min(),  dn_line.min()))
    y_max = float(max(w['High'].max(), up_res.max(), up_line.max()))
    pad = (y_max - y_min) * 0.03 if y_max > y_min else 1.0
    yrange = [y_min - pad, y_max + pad]

    last_label = (w.index[-1].date().isoformat()
                  if isinstance(w.index, pd.DatetimeIndex) else str(w.index[-1]))

    frames.append(go.Frame(
        name=last_label,
        data=[
            go.Candlestick(
                x=w.index, open=w['Open'], high=w['High'], low=w['Low'], close=w['Close'],
                increasing_line_color='rgba(19,156,19,0.8)',
                decreasing_line_color='rgba(175,7,49,0.8)',
                showlegend=False, name='Price'
            ),
            go.Scatter(x=w.index, y=midline, mode="lines",
                       line=dict(color='white', width=0.8), name="MIDLINE"),
            go.Scatter(x=w.index, y=up_res,  mode="lines",
                       line=dict(color='gray', width=1), name="RES_UPPER"),
            go.Scatter(x=w.index, y=dn_res,  mode="lines",
                       line=dict(color='gray', width=1), name="RES_LOWER"),
            # go.Scatter(x=w.index, y=up_line, mode="lines",
            #            line=dict(color='red', width=1), name="STD_UPPER"),
            # go.Scatter(x=w.index, y=dn_line, mode="lines",
            #            line=dict(color='red', width=1), name="STD_LOWER"),
        ],
        layout=go.Layout(
            xaxis=dict(range=[w.index[0], w.index[-1]]),
            yaxis=dict(range=yrange)
        )
    ))

GREEN = "#00DF9A"

fig_anim = go.Figure(
    data=frames[0].data,
    frames=frames,
    layout=go.Layout(
        title="Rolling 86 Day Regression Channel",
        template="plotly_dark",
        width=1000, height=660,
        xaxis=dict(
            rangeslider=dict(visible=False),
            rangebreaks=[dict(bounds=["sat","mon"])],
            gridcolor="#171717"
        ),
        yaxis=dict(autorange=False, gridcolor="#171717"),

        updatemenus=[dict(
            type="buttons",
            direction="right",
            x=0.02, y=1.08, xanchor="left", yanchor="top",
            pad={"r": 10, "t": 0},
            showactive=True,
            bgcolor=GREEN, bordercolor=GREEN, borderwidth=0,
            font=dict(color="black", size=12),
            buttons=[
                dict(label="Play", method="animate",
                     args=[None, {"frame":{"duration":frame_rate,"redraw":True},
                                  "fromcurrent":True,
                                  "transition":{"duration":0},
                                  "mode":"immediate"}]),
                dict(label="Pause", method="animate",
                     args=[[None], {"mode":"immediate",
                                    "frame":{"duration":0,"redraw":False},
                                    "transition":{"duration":0}}]),
            ],
        )],

        sliders=[dict(
            y=-0.08,                         
            pad={"t": 30, "b": 0},           
            currentvalue={
                "prefix": "Date: ",
                "offset": 20,                 
                "font": {"size": 12}
            },
            steps=[dict(
                method="animate",
                args=[[f.name], {"mode":"immediate",
                                 "frame":{"duration":0,"redraw":True},
                                 "transition":{"duration":0}}],
                label=f.name) for f in frames],
        )]
    )
)

fig_anim.show()